In [1]:
from sympy import *
from IPython.display import display, Math, Latex, clear_output, Image
import numpy as np
import cupy as cp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from shapely.geometry import Polygon, box

from IPython.display import display, HTML
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
)) 
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from skimage.feature import peak_local_max
from scipy.ndimage import center_of_mass
from scipy.spatial import Delaunay
from collections import Counter
import os
from pathlib import Path

## 0. Plotting helpers

In [2]:
import re
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc

# --- helpers ---------------------------------------------------------------
def _to_rgba_string(color_str, alpha_override=None):
    s = color_str.strip()
    if s.startswith("#"):
        hexstr = s.lstrip("#")
        if len(hexstr) == 3:
            hexstr = "".join(2*c for c in hexstr)
        r = int(hexstr[0:2], 16)
        g = int(hexstr[2:4], 16)
        b = int(hexstr[4:6], 16)
        a = 1.0
    else:
        m = re.match(
            r"rgba?\s*\(\s*([0-9.+-eE]+)\s*,\s*([0-9.+-eE]+)\s*,\s*([0-9.+-eE]+)(?:\s*,\s*([0-9.+-eE]+))?\s*\)",
            s,
        )
        if not m:
            raise ValueError(f"Unrecognized color format: {color_str!r}")
        r, g, b = [int(float(v)) for v in m.group(1, 2, 3)]
        a = float(m.group(4)) if m.group(4) is not None else 1.0
    if alpha_override is not None:
        a = float(alpha_override)
    return f"rgba({r},{g},{b},{a:.3f})"

def _resolve_plotly_colorscale(name_or_list):
    if isinstance(name_or_list, (list, tuple)):
        return list(name_or_list)
    name = str(name_or_list)
    for ns in ("sequential", "diverging", "cyclical", "qualitative"):
        colset = getattr(pc, ns, None)
        if colset:
            if hasattr(colset, name):
                return getattr(colset, name)
            if name.lower() in getattr(colset, "__dict__", {}):
                return getattr(colset, name.lower())
    if hasattr(pc, name):
        return getattr(pc, name)
    for variant in (name.lower(), name + "_r", name + "-r"):
        for ns in ("sequential", "diverging", "cyclical", "qualitative"):
            colset = getattr(pc, ns, None)
            if colset and hasattr(colset, variant):
                return getattr(colset, variant)
    raise ValueError(f"Could not resolve Plotly colorscale named {name!r}")

# --- public API -----------------------------------------------------------
def get_zcolor(z, colorscale="Rainbow", alpha=1.0, cliplow=0.0, cliphigh=0.0):
    """
    Map numeric array z -> array of rgba strings using a Plotly colorscale.
    - z: 1D numpy array (or array-like)
    - colorscale: Plotly colorscale name or list of colors
    - alpha: float 0..1 to override alpha channel
    Returns: numpy array of shape z.shape with "rgba(r,g,b,a)" strings.
    """
    z = np.asarray(z)
    pl_colors = _resolve_plotly_colorscale(colorscale)
    # normalize z to [0,1] using valid (non-nan) values
    valid = ~np.isnan(z)
    if not valid.any():
        raise ValueError("z contains no valid (non-NaN) values")
    zmin = float(np.nanmin(z))
    zmax = float(np.nanmax(z))
    if zmax == zmin:
        t = np.zeros_like(z, dtype=float)
    else:
        t = ((z - zmin) / (zmax - zmin)) * (1 - cliplow - cliphigh) + cliplow
    # sample colorscale for each normalized value
    sampled = [pc.sample_colorscale(pl_colors, [float(v)])[0] for v in t.flatten()]
    rgba_arr = np.array([_to_rgba_string(s, alpha_override=alpha) for s in sampled])
    return rgba_arr.reshape(z.shape)

def add_colored_line_by_zcolor(
    fig,
    x,
    y,
    z,
    zcolors,
    colorscale="Rainbow",
    colorbar_label="value",
    show_colorbar=True,
    show_line_legend=False,
    line_name=None,
    line_width=3,
    use_scattergl=False,
):
    """
    Add a segmented colored line to fig using precomputed zcolors.
    - fig: plotly.graph_objects.Figure
    - x, y, z: 1D arrays (same length)
    - zcolors: array of rgba strings same shape as z (from get_zcolor)
    - colorscale: used to build the colorbar (name or list)
    - colorbar_label: label for the colorbar
    - show_colorbar: whether to display colorbar
    - show_line_legend: whether to show a single legend entry for the line
    - line_name: legend name (if show_line_legend True)
    - line_width: width of segments
    - use_scattergl: use Scattergl for segment traces
    """
    x = np.asarray(x)
    y = np.asarray(y)
    z = np.asarray(z)
    zcolors = np.asarray(zcolors)
    if not (x.shape == y.shape == z.shape == zcolors.shape):
        raise ValueError("x, y, z, and zcolors must have the same shape")

    Scatter = go.Scattergl if use_scattergl else go.Scatter

    # split at NaNs so we don't draw across gaps
    bad = np.isnan(x) | np.isnan(y) | np.isnan(z)
    if bad.any():
        blocks = []
        start = None
        for i, b in enumerate(bad):
            if not b and start is None:
                start = i
            if (b or i == len(bad) - 1) and start is not None:
                end = i if b else i + 1
                if end - start >= 2:
                    blocks.append((start, end))
                start = None
    else:
        blocks = [(0, len(x))]

    # add one short segment trace per pair of points
    for (start, end) in blocks:
        for i in range(start, end - 1):
            color = str(zcolors[i])
            showleg = show_line_legend and (i == start) and (line_name is not None)
            name = line_name if showleg else None
            fig.add_trace(
                Scatter(
                    x=[x[i], x[i + 1]],
                    y=[y[i], y[i + 1]],
                    mode="lines",
                    line=dict(color=color, width=line_width),
                    # hoverinfo="skip",
                    showlegend=showleg,
                    name=name,
                )
            )

    # invisible scatter to provide colorbar (maps z -> colorscale)
    if show_colorbar:
        pl_colors = _resolve_plotly_colorscale(colorscale)
        n = len(pl_colors)
        plotly_colorscale = [[i / (n - 1), pl_colors[i]] for i in range(n)]
        z_valid = z[~np.isnan(z)]
        zmin = float(np.nanmin(z_valid))
        zmax = float(np.nanmax(z_valid))
        fig.add_trace(
            Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(
                    color=z_valid,
                    colorscale=plotly_colorscale,
                    cmin=zmin,
                    cmax=zmax,
                    size=0.1,
                    showscale=True,
                    colorbar=dict(title=colorbar_label),
                ),
                hoverinfo="none",
                showlegend=False,
                opacity=0,
            )
        )
    return fig

In [3]:
def Plot(sim, Show=True, file=None, zrange=None, profileLine=None):
  if zrange is not None:
     zmin=zrange[0]
     zmax=zrange[1]
  else:
     zmin=sim.phi.get().min()
     zmax=sim.phi.get().max()
  _fig = go.Figure(data=go.Heatmap(x=np.arange(sim.nx)*sim.dx,
                                  y=np.arange(sim.ny)*sim.dy,
                                  z=sim.phi.get(),
                                  zmin=zmin,
                                  zmax=zmax,
                                  colorscale='gray',
                                  showscale=False))

  # profile line, if specified
  if profileLine is not None:
    _fig.add_trace(go.Scatter(
      x = profileLine["x"],
      y = profileLine["y"],
      mode = "lines",
      line_color = profileLine["color"]
    ))

  # decide pixels per cell (1 means 1 _figure pixel per heatmap cell)
  px_per_cell = 2

  # exact _figure pixel size to match heatmap grid
  _fig_width = int(sim.nx * px_per_cell)
  _fig_height = int(sim.ny * px_per_cell)

  _fig.update_layout(
      width=_fig_width,
      height=_fig_height,
      autosize=False,
      margin=dict(l=0, r=0, t=0, b=0),
      plot_bgcolor='white',
      # place colorbar inside plot or hide it to avoid extra space
      # if you want a colorbar keep it small and place it on top of the image
      # e.g. _fig.data[0].colorbar.update({'len':1, 'thickness':8, 'x':0.995})
  )

  _fig.update_xaxes(
      showgrid=False, showticklabels=False,
      # force axis to use full horizontal domain
      domain=[0.0, 1.0],
      range=[0, sim.nx * sim.dx]   # match the heatmap coordinates
  )

  _fig.update_yaxes(
      showgrid=False, showticklabels=False,
      domain=[0.0, 1.0],
      range=[0, sim.ny * sim.dy],  # match the heatmap coordinates
      scaleanchor='x',             # lock aspect ratio so a unit in x equals unit in y
      scaleratio=1
  )

  # optionally hide the colorbar to remove its margin impact
  if hasattr(_fig.data[0], 'colorbar'):
      _fig.data[0].colorbar.update({'thickness': 8, 'len': 1, 'x': 0.995})  # place thin bar on right
      # or remove it: _fig.data[0].colorbar = dict() or _fig.data[0].showscale = False


  _annotations=[
      dict(
        text=f"t = {sim.t:.2f}",
        showarrow=False,
        xref="paper", yref="paper",
        x=0.05, y=0.9,
        xanchor="left", yanchor="top",
        font=dict(color="cyan", size=16, family="Arial"),
      ),
      dict(
        text=f"f = {sim.f:.8f}",
        showarrow=False,
        xref="paper", yref="paper",
        x=0.95, y=0.1,
        xanchor="right", yanchor="bottom",
        font=dict(color="yellow", size=16, family="Arial"),
      )
    ]

  _fig.update_layout(
    annotations = _annotations
  )

  if Show:
    _fig.show()

  if file is not None:
    _fig.write_image(file, scale=2)
  
  # return _fig

In [4]:
def plot_energy_relaxation(ft, f_t, epsilon=0.0, beta_val=0.0, g_val=0.0, type="Log"):
  # compute dfdt for coloring
  dfdt = np.log10(np.abs(np.gradient(f_t, ft)) + np.abs(np.gradient(f_t, ft))[np.abs(np.gradient(f_t, ft)) > 0].min())

  # only show data after t_min
  t_min = 0.075
  _x = ft[ft >= t_min * ft.max()]
  _y = f_t[ft >= t_min * ft.max()]
  _dfdt = dfdt[ft >= t_min * ft.max()]

  # choose colorscale and get colors for each z
  colorscale_name = "Rainbow"
  zcolors = get_zcolor(_dfdt, colorscale=colorscale_name, alpha=1.0)

  # build figure and add colored line
  fig = go.Figure()
  add_colored_line_by_zcolor(
      fig,
      _x,
      _y,
      _dfdt,
      zcolors,
      colorscale=colorscale_name,
      colorbar_label="log(|dy/dx|)",
      show_colorbar=True,
      show_line_legend=False,
      line_name="signal",
      line_width=3,
      use_scattergl=True,
  )

  # annotations at specified t positions
  annot_t_times = [0.2, 0.5, 1.0]
  annot_t_positioning = [(40, -40, "left", "bottom"),
                        (0, -40, "center", "bottom"),
                        (-40, -40, "right", "bottom")]
  annotations = []
  for i, tpos in enumerate(annot_t_times):
      idx = int(np.argmin(np.abs(_x - tpos * ft.max())))
      x_pt = float(_x[idx])
      y_pt = float(_y[idx])
      z_val = float(_dfdt[idx])
      color_rgba = str(zcolors[idx])  # same color used on the line

      text = f"t: {x_pt:.6g}<br>f(t): {y_pt:.6g}<br>|df/dt|: {10**z_val:.6g}"
      ann = dict(
          x=x_pt,
          y=y_pt,
          xref="x",
          yref="y",
          text=text,
          showarrow=True,
          arrowcolor=color_rgba,
          font=dict(color=color_rgba),
          ax=annot_t_positioning[i][0],   # text offset to the right (pixels)
          ay=annot_t_positioning[i][1],   # text offset above (pixels)
          xanchor=annot_t_positioning[i][2],
          yanchor=annot_t_positioning[i][3],
          align="left",
          arrowwidth=2,
          arrowhead=2,
          bgcolor="rgba(255,255,255,0.85)",
      )
      annotations.append(ann)

  # final layout update (y range, axis labels, annotations)
  fig.update_layout(
    width=800,
    height=600,
    title=dict(
        text=r'$\Large{\mathsf{Random\ Field\ Relaxation}\\ \small\mathsf{' + type + r'\ PFC:\ }(\epsilon=' + f'{epsilon:.3f},' + r'\beta=' + f'{beta_val:.3f},' + r'g=' + f'{g_val:.3f}' + r')}$',
        x=0.5,
        xanchor="center",
        font=dict(family="Arial", size=20, color="black")
    ),
    plot_bgcolor='white',          # plot area background
    paper_bgcolor='white',         # surrounding page background
    xaxis=dict(title="t", range=[ft[0], ft[-1]]),
    yaxis=dict(title="f(t)"),
    annotations=annotations,
    margin=dict(l=60, r=20, t=120, b=60),
  )

  light_gray = '#e6e6e6'  # subtle light gray (adjust hex to taste)
  fig.update_xaxes(
      showgrid=True, gridcolor=light_gray, gridwidth=1,
      zeroline=True, zerolinecolor=light_gray,
      # dtick=1,
      linecolor='black', mirror=True, ticks='outside'
  )
  fig.update_yaxes(
      showgrid=True, gridcolor=light_gray, gridwidth=1,
      zeroline=True, zerolinecolor=light_gray,
      # dtick=0.1,
      linecolor='black', mirror=True, ticks='outside'
  )

  return fig

## 1. Read Data

In [5]:
outputRoot = Path('.')

In [6]:
# read data from 1x-vacancy-log.npz
data = np.load(outputRoot / f"1x-vacancy-log.npz")
mx = data['mx']
x = data['x']
y = data['y']
log_phi_1x = data['phi']
data = np.load(outputRoot / f"2x-vacancy-log.npz")
log_phi_2x = data['phi']
data = np.load(outputRoot / f"3x-vacancy-log.npz")
log_phi_3x = data['phi']
data = np.load(outputRoot / f"1x-vacancy-std.npz")
std_phi_1x = data['phi']
data = np.load(outputRoot / f"2x-vacancy-std.npz")
std_phi_2x = data['phi']
data = np.load(outputRoot / f"3x-vacancy-std.npz")
std_phi_3x = data['phi']


## 2. Plot

In [10]:
fig = go.Figure()
markers=['triangle-up', 'hexagon', 'diamond', 'cross']
_slices = np.arange(-9,-4,2)
_colors = get_zcolor(np.linspace(0,1,3), 'Bluered', alpha=0.5)
_a0 = x.max() / mx

fig.add_trace(go.Scatter(x=x[90,:]/_a0, y=log_phi_1x[90,:] - 0, mode='lines', line=dict(color=_colors[0]), name='1x Vacancy'))
fig.add_trace(go.Scatter(x=x[90,:]/_a0, y=log_phi_2x[90,:] - 2, mode='lines', line=dict(color=_colors[1]), name='2x Vacancy'))
fig.add_trace(go.Scatter(x=x[90,:]/_a0, y=log_phi_3x[90,:] - 4, mode='lines', line=dict(color=_colors[2]), name='3x Vacancy'))

fig.update_layout(
  plot_bgcolor='white',
  paper_bgcolor='white',
  width=1200, height=400,
  xaxis=dict(range=[x[90,0]/_a0, x[90,-1]/_a0]),
  margin=dict(l=5, r=5, t=5, b=5),
  legend=dict(
      x=0.02,               # horizontal position (0 = left, 1 = right)
      y=0.98,               # vertical position (0 = bottom, 1 = top)
      xanchor='left',       # anchor legend's x to its left side
      yanchor='top',        # anchor legend's y to its top
      bgcolor='rgba(255,255,255,0.9)',  # semi-transparent background
      bordercolor='black',
      borderwidth=1,
      orientation='v'       # 'v' for vertical, 'h' for horizontal
  ),
)

light_gray = '#e6e6e6'  # subtle light gray (adjust hex to taste)
fig.update_xaxes(
    showgrid=True, gridcolor=light_gray, gridwidth=1,
    zeroline=True, zerolinecolor=light_gray,
    dtick=1,
    tickmode='array',
    tickvals=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    ticktext=['$0$', '$a_0$', '$2 a_0$', '$3 a_0$', '$4 a_0$', '$5 a_0$', '$6 a_0$', '$7 a_0$', '$8 a_0$', '$9 a_0$'],
    linecolor='black', mirror=True, ticks='outside'
)
fig.update_yaxes(
    showgrid=True, gridcolor=light_gray, gridwidth=1,
    zeroline=False, zerolinecolor=light_gray,
    tickmode='array',
    ticks='',
    tickvals=[0, -2],
    ticktext=['', ''],
    linecolor='black', mirror=True
)

fig.show()
fig.write_image(outputRoot / f'fig-1x2x3x-vacancy-log-density-profile.png', scale=2)

In [ ]:
Plot(sim_log, Show=True, file=outputRoot / "stable" / f"fig-1x-vacancy-log-groundstate.png", zrange=None,
    profileLine={
      "x": [0, sim_log.x.get()[90,:].max()],
      "y": [sim_log.y.get()[90,0], sim_log.y.get()[90,-1]],
      "color": _colors[-2]
      })

In [11]:
fig = go.Figure()
markers=['triangle-up', 'hexagon', 'diamond', 'cross']
_slices = np.arange(-9,-4,2)
_colors = get_zcolor(np.linspace(0,1,3), 'Bluered', alpha=0.5)
_a0 = x.max() / mx

fig.add_trace(go.Scatter(x=x[90,:]/_a0, y=std_phi_1x[90,:]-std_phi_1x[90,:].min() - 0, mode='lines', line=dict(color=_colors[0]), name='1x Vacancy'))
fig.add_trace(go.Scatter(x=x[90,:]/_a0, y=std_phi_2x[90,:]-std_phi_1x[90,:].min() - 2, mode='lines', line=dict(color=_colors[1]), name='2x Vacancy'))
fig.add_trace(go.Scatter(x=x[90,:]/_a0, y=std_phi_3x[90,:]-std_phi_1x[90,:].min() - 4, mode='lines', line=dict(color=_colors[2]), name='3x Vacancy'))

fig.update_layout(
  plot_bgcolor='white',
  paper_bgcolor='white',
  width=1200, height=400,
  xaxis=dict(range=[x[90,0]/_a0, x[90,-1]/_a0]),
  margin=dict(l=5, r=5, t=5, b=5),
  legend=dict(
      x=0.02,               # horizontal position (0 = left, 1 = right)
      y=0.98,               # vertical position (0 = bottom, 1 = top)
      xanchor='left',       # anchor legend's x to its left side
      yanchor='top',        # anchor legend's y to its top
      bgcolor='rgba(255,255,255,0.9)',  # semi-transparent background
      bordercolor='black',
      borderwidth=1,
      orientation='v'       # 'v' for vertical, 'h' for horizontal
  ),
)

light_gray = '#e6e6e6'  # subtle light gray (adjust hex to taste)
fig.update_xaxes(
    showgrid=True, gridcolor=light_gray, gridwidth=1,
    zeroline=True, zerolinecolor=light_gray,
    dtick=1,
    tickmode='array',
    tickvals=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    ticktext=['$0$', '$a_0$', '$2 a_0$', '$3 a_0$', '$4 a_0$', '$5 a_0$', '$6 a_0$', '$7 a_0$', '$8 a_0$', '$9 a_0$'],
    linecolor='black', mirror=True, ticks='outside'
)
fig.update_yaxes(
    showgrid=True, gridcolor=light_gray, gridwidth=1,
    zeroline=False, zerolinecolor=light_gray,
    tickmode='array',
    ticks='',
    tickvals=[0, -2],
    ticktext=['', ''],
    linecolor='black', mirror=True
)

fig.show()
fig.write_image(outputRoot / f'fig-1x2x3x-vacancy-std-density-profile.png', scale=2)